In [ ]:
import json
with open('data.json','rb') as f:
    data=json.load(f)

In [ ]:
print(data[0].keys())

dict_keys(['text', 'entities'])


In [ ]:

import spacy
from spacy.training import Example
from spacy.util import minibatch, compounding
import random
import json
from spacy.util import filter_spans
with open('data.json','rb') as f:
    train_data=json.load(f)
# create a blank English NLP model
nlp = spacy.blank('en')

# Create the NER component and add it to the pipeline
if "ner" not in nlp.pipe_names:
    ner = nlp.add_pipe("ner", last=True)
else:
    ner = nlp.get_pipe("ner")

# Add labels to the NER component
for item in train_data:
    for _, _, label in item['entities']:
        ner.add_label(label)

# Prepare training data in the format required by spaCy 3.x
train_examples = []
count=0
for item in train_data:
    doc = nlp.make_doc(item["text"])
    ents = []
    for start, end, label in item['entities']:
        span = doc.char_span(start, end, label=label, alignment_mode="contract")
        if span is not None:
            ents.append(span)

    filtered_ents = filter_spans(ents)
    doc.ents = filtered_ents
    example = Example.from_dict(doc, {"entities": item['entities']})
    train_examples.append(example)
# Initialize the optimizer
optimizer = nlp.begin_training()

# Training loop
n_iter = 300
for itn in range(n_iter):
    random.shuffle(train_examples)
    losses = {}
    # Batch up the examples using spaCy's minibatch
    batches = minibatch(train_examples, size=compounding(4.0, 32.0, 1.001))
    for batch in batches:
        nlp.update(
            batch,  # batch of Example objects
            drop=0.2,  # dropout - make it harder to memorise data
            sgd=optimizer,  # callable to update weights
            losses=losses
        )
    scores = nlp.evaluate(train_examples)
    ents_p = scores["ents_p"]
    ents_r = scores["ents_r"]
    ents_f = scores["ents_f"]

    print(f"Iteration {itn+1}: Losses: {losses['ner']:.3f}, Precision: {ents_p:.3f}, Recall: {ents_r:.3f}, F1-score: {ents_f:.3f}")

# Save the model
nlp.to_disk("ner_model")


Iteration 1: Losses: 26291.029, Precision: 1.000, Recall: 1.000, F1-score: 1.000
Iteration 2: Losses: 6220.252, Precision: 0.965, Recall: 1.000, F1-score: 0.982
Iteration 3: Losses: 4165.253, Precision: 0.940, Recall: 1.000, F1-score: 0.969
Iteration 4: Losses: 3254.169, Precision: 0.823, Recall: 1.000, F1-score: 0.903
Iteration 5: Losses: 2816.303, Precision: 0.922, Recall: 1.000, F1-score: 0.959
Iteration 6: Losses: 2445.728, Precision: 0.795, Recall: 1.000, F1-score: 0.886
Iteration 7: Losses: 2289.312, Precision: 0.926, Recall: 1.000, F1-score: 0.962
Iteration 8: Losses: 2112.044, Precision: 0.904, Recall: 1.000, F1-score: 0.950
Iteration 9: Losses: 1926.784, Precision: 0.897, Recall: 1.000, F1-score: 0.946
Iteration 10: Losses: 1884.441, Precision: 0.876, Recall: 1.000, F1-score: 0.934
Iteration 11: Losses: 1784.231, Precision: 0.874, Recall: 1.000, F1-score: 0.933
Iteration 12: Losses: 1578.328, Precision: 0.892, Recall: 1.000, F1-score: 0.943
Iteration 13: Losses: 1557.933, Prec

KeyboardInterrupt: 

In [ ]:
# Print the entities of the first item in your data to see the true structure
print(data[0]['entities'][0])

[1296, 1622, 'SKILLS']


In [ ]:
import re

train_data = []

for i in data:
    item = {}
    entities = []
    content = i['text']

    # Track occupied character spans to prevent overlaps
    occupied_spans = []


    item['text'] = content
    item['entities'] = entities
    train_data.append(item)

    for j in i['entities']:
        # Format the label (e.g., "Job Title" -> "JOB_TITLE")
        label = j[2][0].replace(" ", "_").upper()
        text = content.strip()

        # Escape the text in case it contains regex special characters (like . or *)
        escaped_text = re.escape(text)

        # Find ALL occurrences of the text in the document
        for match in re.finditer(escaped_text, content):
            start = match.start()
            end = match.end()

            # Check for overlaps with already saved entities
            is_overlapping = False
            for span in occupied_spans:
                span_start, span_end = span
                # Simple math to check if two ranges overlap
                if max(start, span_start) < min(end, span_end):
                    is_overlapping = True
                    break

            # If there is no overlap, save it!
            if not is_overlapping:
                # Using tuples instead of lists for standard NER formatting
                entities.append((start, end, label))
                occupied_spans.append((start, end))

    # Sort entities by their start position (Highly recommended for spaCy)
    entities.sort(key=lambda x: x[0])

    item['text'] = content
    item['entities'] = entities
    train_data.append(item)

In [ ]:
with open('data.json','w') as file:
    json.dump(train_data,file)